In [ ]:
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import torch

# 模型合并

In [ ]:
 # 加载基础模型
from peft import PeftModel


base_model = AutoModelForCausalLM.from_pretrained(
                "/mnt/project/deepseek-r1-distill-llama-8b-qlora/model/DeepSeek-R1-Distill-Llama-8B",
                torch_dtype=torch.bfloat16,
                quantization_config=None
            )    
# 加载适配器
merged_model = PeftModel.from_pretrained(
                base_model,
                "/mnt/project/deepseek-r1-distill-llama-8b-qlora/output/final_model/lora_adapters",
                is_trainable=False
            )  
# 合并权重
merged_model = merged_model.merge_and_unload()

# 保存合并模型
merged_model.save_pretrained(
    "/mnt/project/deepseek-r1-distill-llama-8b-qlora/output/DeepSeek-R1-Distill-Llama-8B-Qlora",
    safe_serialization=True,
    max_shard_size="100GB"
) 


In [ ]:
base_model_path ='/mnt/project/deepseek-r1-distill-llama-8b-qlora/model/DeepSeek-R1-Distill-Llama-8B'
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    max_seq_length=2048,
    padding_side="right",
    use_fast=True
)

# 确保设置 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 添加量化配置对象（更规范）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 加载模型
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    use_cache=False,
)

In [ ]:
model_path ='/mnt/project/deepseek-r1-distill-llama-8b-qlora/output/DeepSeek-R1-Distill-Llama-8B-Qlora'
# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    max_seq_length=2048,
    padding_side="right",
    use_fast=True
)

# 确保设置 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 添加量化配置对象（更规范）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    use_cache=False,
)

In [ ]:
test_dataset = load_from_disk("/mnt/project/deepseek-r1-distill-llama-8b-qlora/data/test_dataset")

In [ ]:
test_dataset[12:14]

In [ ]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. Please answer the following medical question.

### Question:
{}

### Response:
<think>{}"""


In [ ]:
question = "一位72岁的男性患者，有20年的高血压病史，现在在全麻复合硬膜外麻醉下接受胆囊手术后，哪种镇痛方法最为合适？"

model.eval()
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1024,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])

# 原始模型推理

In [ ]:
base_model.eval()
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = base_model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1024,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])